Instalación de librerías necesarias para procesamiento de datos, generación de embeddings y construcción de la base vectorial.



In [1]:
!pip install sentence-transformers faiss-cpu pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 38.9 MB/s eta 0:00:00


Importación de librerías

In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import faiss

Dataset Sobre Customer Support Tickets



In [6]:
from google.colab import files

uploaded = files.upload()

Saving customer_support_tickets.csv to customer_support_tickets.csv


In [7]:
import pandas as pd

nombre_archivo = list(uploaded.keys())[0]

df = pd.read_csv(nombre_archivo)

print(df.shape)

(8469, 17)


Agente 1: Normalizador

Carga el dataset.
Detecta valores nulos.
Detecta duplicados.
Limpia texto.
Genera un dataset limpio.

In [8]:
import pandas as pd

class AgenteNormalizador:

    def __init__(self, archivo):
        self.archivo = archivo

    def ejecutar(self):

        df = pd.read_csv(self.archivo)

        print("===== ANALISIS INICIAL =====")
        print("Filas y columnas:", df.shape)

        print("\nValores nulos:")
        print(df.isnull().sum())

        print("\nDuplicados:")
        print(df.duplicated().sum())

        # Eliminar duplicados
        df = df.drop_duplicates()

        # Completar valores faltantes
        for columna in df.columns:

            if df[columna].dtype == "object":
                df[columna] = df[columna].fillna("No especificado")
            else:
                df[columna] = df[columna].fillna(df[columna].median())

        # Normalizar texto
        for columna in df.select_dtypes(include="object").columns:

            df[columna] = (
                df[columna]
                .astype(str)
                .str.lower()
                .str.strip()
            )

        print("\n===== DATASET NORMALIZADO =====")
        print(df.shape)

        return df


agente_normalizador = AgenteNormalizador(nombre_archivo)

df_limpio = agente_normalizador.ejecutar()

===== ANALISIS INICIAL =====
Filas y columnas: (8469, 17)

Valores nulos:
Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                      5700
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64

Duplicados:
0

===== DATASET NORMALIZADO =====
(8469, 17)


In [9]:
df_limpio.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,marisa obrien,carrollallison@example.com,32,other,gopro hero,2021-03-22,technical issue,product setup,i'm having an issue with the {product_purchase...,pending customer response,no especificado,critical,social media,2023-06-01 12:15:36,no especificado,3.0
1,2,jessica rios,clarkeashley@example.com,42,female,lg smart tv,2021-05-22,technical issue,peripheral compatibility,i'm having an issue with the {product_purchase...,pending customer response,no especificado,critical,chat,2023-06-01 16:45:38,no especificado,3.0
2,3,christopher robbins,gonzalestracy@example.com,48,other,dell xps,2020-07-14,technical issue,network problem,i'm facing a problem with my {product_purchase...,closed,case maybe show recently my computer follow.,low,social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,christina dillon,bradleyolson@example.org,27,female,microsoft office,2020-11-13,billing inquiry,account access,i'm having an issue with the {product_purchase...,closed,try capital clearly never color toward story.,low,social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,alexander carroll,bradleymark@example.com,67,female,autodesk autocad,2020-02-04,billing inquiry,data loss,i'm having an issue with the {product_purchase...,closed,west decision evidence bit.,low,email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


Agente 2: Entrenador

Recibe el dataset limpio.
Genera embeddings con un Transformer.
Construye la base vectorial FAISS.

Crear texto para los embeddings

In [10]:
df_limpio["texto_completo"] = (
    df_limpio["Ticket Subject"] + " " +
    df_limpio["Ticket Description"]
)

df_limpio["texto_completo"].head()

,texto_completo
0,product setup i'm having an issue with the {pr...
1,peripheral compatibility i'm having an issue w...
2,network problem i'm facing a problem with my {...
3,account access i'm having an issue with the {p...
4,data loss i'm having an issue with the {produc...


Instalar librerías del Agente Entrenador

In [11]:
!pip install sentence-transformers faiss-cpu -q

Importar librerías

In [12]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

Cargar Transformer

In [13]:
modelo = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Transformer cargado correctamente")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Transformer cargado correctamente


Generar embeddings

In [14]:
embeddings = modelo.encode(
    df_limpio["texto_completo"].tolist(),
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/265 [00:00<?, ?it/s]

(8469, 384)


Crear Base Vectorial

In [15]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("Vectores almacenados:")
print(index.ntotal)

Vectores almacenados:
8469


Implementar búsqueda RAG

In [16]:
consulta = "cannot login to my account"

vector_consulta = modelo.encode([consulta])

distancias, indices = index.search(
    np.array(vector_consulta).astype("float32"),
    5
)

print(indices)

[[6600 1816  160 3386  640]]


Mostrar resultados recuperados

In [17]:
for i in indices[0]:

    print("="*60)

    print("TIPO:")
    print(df_limpio.iloc[i]["Ticket Type"])

    print("\nASUNTO:")
    print(df_limpio.iloc[i]["Ticket Subject"])

    print("\nDESCRIPCION:")
    print(df_limpio.iloc[i]["Ticket Description"])

TIPO:
cancellation request

ASUNTO:
account access

DESCRIPCION:
i'm facing issues logging into my {product_purchased} account. it says my account is locked. what should i do to unlock it?

you should open the account immediately and log in, either with username/password or on i've followed online tutorials and community forums to troubleshoot the issue, but no luck so far.
TIPO:
product inquiry

ASUNTO:
account access

DESCRIPCION:
i'm facing issues logging into my {product_purchased} account. it says my account is locked. what should i do to unlock it?

the first step is to create a new account using your user name or a password you i've tried troubleshooting steps mentioned in the user manual, but the issue persists.
TIPO:
refund request

ASUNTO:
account access

DESCRIPCION:
i'm facing issues logging into my {product_purchased} account. it says my account is locked. what should i do to unlock it?

open the account. you'll find everything you need to do. the next step is i've reviewe

Agente 3: Comunicador
Recupera información relevante.
Genera un reporte en lenguaje natural.

In [18]:
class AgenteComunicador:

    def generar_reporte(self, df):

        print("===== REPORTE GENERAL =====")

        print("\nCantidad total de tickets:")
        print(len(df))

        print("\nTipos de ticket:")
        print(df["Ticket Type"].value_counts())

        print("\nPrioridades:")
        print(df["Ticket Priority"].value_counts())

        print("\nEstados:")
        print(df["Ticket Status"].value_counts())


comunicador = AgenteComunicador()

comunicador.generar_reporte(df_limpio)

===== REPORTE GENERAL =====

Cantidad total de tickets:
8469

Tipos de ticket:
Ticket Type
refund request          1752
technical issue         1747
cancellation request    1695
product inquiry         1641
billing inquiry         1634
Name: count, dtype: int64

Prioridades:
Ticket Priority
medium      2192
critical    2129
high        2085
low         2063
Name: count, dtype: int64

Estados:
Ticket Status
pending customer response    2881
open                         2819
closed                       2769
Name: count, dtype: int64
